In [30]:
import numpy as np
import pandas as pd

df_cleaned = pd.read_csv("../../data/cleaned/Metallic Glass Transition Data/Metallic_Glass_Forming_with_features_Matminer_Features.csv")
df_cleaned.head()

,Material compositions,Trg,Density_composition_average,IsBoron_composition_average,IsDBlock_composition_average,NdValence_composition_average,NValance_composition_average,HeatVaporization_max_value,BoilingT_difference,HeatVaporization_difference,...,Ni,Ag,C,Number_elements,Atomic_size_mismatch,Mixing_enthalpy,Mixing_entropy,Formation_amorphous,Topological_formation,Band sizes
0,Ag10Cu35Zr55,0.534,0.579069,-0.5856,1.029882,0.589536,0.217571,0.563915,0.544521,0.505748,...,0.0,10.0,0.0,3,0.067424,-21.83,0.007703,-22.917082,6.185585,3.925308
1,Ag10Cu50Zr40,0.608,0.748401,-0.5856,1.029882,1.176250,0.653969,0.563915,0.544521,0.505748,...,0.0,10.0,0.0,3,0.072817,-21.20,0.007843,-26.842852,5.782700,4.067730
2,Ag10Cu55Zr35,0.609,0.804845,-0.5856,1.029882,1.371822,0.799435,0.563915,0.544521,0.505748,...,0.0,10.0,0.0,3,0.073320,-20.07,0.007703,-26.349847,5.648405,4.116343
3,Ag20Cu40Zr40,0.577,0.821973,-0.5856,1.029882,1.176250,0.653969,0.563915,0.544521,0.505748,...,0.0,20.0,0.0,3,0.072772,-20.48,0.008771,-23.422410,5.739440,4.064280
4,Ag35Ca65,0.515,-0.860992,-0.5856,-0.998961,-0.437214,-0.613663,-1.684370,-1.368929,-1.228445,...,0.0,35.0,0.0,2,0.055141,-25.48,0.005383,-31.314943,4.041450,3.492169


In [31]:
import pycaret
pycaret.__version__

'3.3.2'

In [50]:
from pycaret.regression import *
# Keep "Material compositions" column for later
material_compositions = df_cleaned.iloc[:, 0]

df_features = df_cleaned.iloc[:, 1:]
s = setup(data=df_features, target="Trg", normalize=True, train_size=0.8, session_id=123)
# s = setup(data=df_features, target="Trg", session_id=123)

# s = setup(
#     data=df_features,
#     target="Trg",
#     low_variance_threshold=1e-4,
#     normalize=True,
#     fold=10,
#     train_size=0.8,
#     n_jobs=-1,
#     session_id=3131,
#     remove_multicollinearity=True,
#     multicollinearity_threshold=0.6,
#     transform_target=True,
#     remove_outliers=True,
#     outliers_threshold=0.05,
#     polynomial_features=True,
# )

,Description,Value
0,Session id,123
1,Target,Trg
2,Target type,Regression
3,Original data shape,"(537, 54)"
4,Transformed data shape,"(537, 54)"
5,Transformed train set shape,"(429, 54)"
6,Transformed test set shape,"(108, 54)"
7,Numeric features,53
8,Preprocess,True
9,Imputation type,simple


In [51]:
best = compare_models(n_select=3)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.0160,0.0006,0.0245,0.6432,0.0158,0.0288,0.0290
rf,Random Forest Regressor,0.0170,0.0007,0.0249,0.6327,0.0161,0.0306,0.0430
lightgbm,Light Gradient Boosting Machine,0.0179,0.0007,0.0256,0.6128,0.0165,0.0320,0.0440
gbr,Gradient Boosting Regressor,0.0191,0.0007,0.0264,0.5894,0.0170,0.0342,0.0240
knn,K Neighbors Regressor,0.0212,0.0009,0.0294,0.4926,0.0189,0.0379,0.0050
ada,AdaBoost Regressor,0.0247,0.0010,0.0316,0.4092,0.0203,0.0437,0.0170
br,Bayesian Ridge,0.0243,0.0011,0.0324,0.3820,0.0208,0.0434,0.0050
ridge,Ridge Regression,0.0243,0.0012,0.0332,0.3426,0.0213,0.0433,0.0050
dt,Decision Tree Regressor,0.0225,0.0012,0.0345,0.2918,0.0224,0.0402,0.0060
huber,Huber Regressor,0.0237,0.0012,0.0341,0.2850,0.0219,0.0423,0.0080


In [34]:
best

[ExtraTreesRegressor(n_jobs=-1, random_state=123),
 RandomForestRegressor(n_jobs=-1, random_state=123),
 LGBMRegressor(n_jobs=-1, random_state=123)]

In [35]:
# et_model = best[0]

# plot_model(et_model, plot="rfe")

In [36]:
tuned_models = [
    tune_model(model) for model in best
]

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0199,0.0009,0.0301,0.3999,0.0191,0.0346
1,0.0215,0.0010,0.0310,0.4015,0.0200,0.0389
2,0.0229,0.0009,0.0295,0.3211,0.0187,0.0398
3,0.0289,0.0014,0.0369,0.1851,0.0236,0.0513
4,0.0222,0.0009,0.0295,0.2829,0.0186,0.0380
5,0.0266,0.0015,0.0385,0.2628,0.0250,0.0488
6,0.0271,0.0012,0.0349,0.2952,0.0224,0.0487
7,0.0331,0.0022,0.0474,0.0974,0.0310,0.0627
8,0.0293,0.0013,0.0363,0.2063,0.0232,0.0525


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0179,0.0007,0.0262,0.5472,0.0165,0.0309
1,0.0197,0.0007,0.0266,0.5583,0.0171,0.0352
2,0.0196,0.0007,0.0261,0.4696,0.0165,0.0339
3,0.0263,0.0011,0.0333,0.3389,0.0212,0.0462
4,0.0204,0.0007,0.0267,0.4128,0.0168,0.0346
5,0.0236,0.0012,0.0347,0.4012,0.0226,0.0432
6,0.0261,0.0012,0.0343,0.3190,0.0220,0.0466
7,0.0302,0.0020,0.0447,0.1956,0.0293,0.0573
8,0.0265,0.0010,0.0321,0.3776,0.0206,0.0472


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0296,0.0016,0.0395,-0.0330,0.0250,0.0506
1,0.0315,0.0016,0.0400,0.0010,0.0257,0.0560
2,0.0294,0.0013,0.0357,0.0068,0.0226,0.0508
3,0.0340,0.0017,0.0411,-0.0064,0.0262,0.0596
4,0.0261,0.0012,0.0351,-0.0169,0.0221,0.0445
5,0.0351,0.0020,0.0451,-0.0092,0.0290,0.0630
6,0.0335,0.0017,0.0418,-0.0097,0.0268,0.0600
7,0.0359,0.0025,0.0500,-0.0044,0.0326,0.0675
8,0.0347,0.0017,0.0407,0.0011,0.0260,0.0614


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [37]:
# et = create_model('et', return_train_score=True, max_depth=20, n_estimators=50, min_samples_split=2)

In [38]:
# plot_model(et, plot="residuals")

In [39]:
et_model = create_model('et', return_train_score=True)

In [14]:
rand_search = {
    'n_estimators': (100, 600),
    'max_depth': (5, 20),
    'min_samples_split': (2, 20),
    'min_samples_leaf': (1, 10),
    'max_features': (0.5, 1.0),
    'bootstrap': [True, False]
}

tuned_random = tune_model(
    et_model,
    optimize='R2',
    n_iter=50,                 # 50 random trials
    search_library='optuna',
    search_algorithm='tpe',
    custom_grid=rand_search
)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0147,0.0004,0.0211,0.7050,0.0133,0.0253
1,0.0118,0.0004,0.0189,0.7778,0.0118,0.0205
2,0.0135,0.0005,0.0213,0.6453,0.0135,0.0237
3,0.0211,0.0009,0.0296,0.4755,0.0190,0.0368
4,0.0108,0.0002,0.0156,0.8000,0.0098,0.0185
5,0.0144,0.0006,0.0235,0.7250,0.0155,0.0267
6,0.0184,0.0008,0.0285,0.5281,0.0185,0.0335
7,0.0218,0.0014,0.0379,0.4220,0.0251,0.0426
8,0.0165,0.0006,0.0243,0.6439,0.0157,0.0295


In [15]:
best_params = tuned_random.get_params()
# Narrow ranges for grid search based on best params
grid_search = {
    'n_estimators': [max(50, best_params['n_estimators']-100), best_params['n_estimators'], best_params['n_estimators']+100],
    'max_depth': [max(3, best_params['max_depth']-3), best_params['max_depth'], best_params['max_depth']+3],
    'min_samples_split': [max(2, best_params['min_samples_split']-3), best_params['min_samples_split'], best_params['min_samples_split']+3],
    'min_samples_leaf': [max(1, best_params['min_samples_leaf']-2), best_params['min_samples_leaf'], best_params['min_samples_leaf']+2],
    'max_features': [max(0.3, best_params['max_features']-0.1), best_params['max_features'], min(1.0, best_params['max_features']+0.1)],
    'bootstrap': [best_params['bootstrap']]
}

In [16]:
tuned_grid = tune_model(
    et_model,
    optimize='R2',
    search_library='scikit-learn',
    search_algorithm='grid',
    custom_grid=grid_search
)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0147,0.0004,0.0212,0.7038,0.0134,0.0252
1,0.0116,0.0003,0.0187,0.7829,0.0117,0.0201
2,0.0135,0.0004,0.0210,0.6562,0.0133,0.0236
3,0.0208,0.0009,0.0292,0.4920,0.0187,0.0362
4,0.0109,0.0003,0.0158,0.7936,0.0100,0.0187
5,0.0145,0.0006,0.0235,0.7253,0.0155,0.0269
6,0.0182,0.0008,0.0285,0.5308,0.0184,0.0332
7,0.0218,0.0015,0.0381,0.4153,0.0253,0.0425
8,0.0162,0.0006,0.0240,0.6530,0.0156,0.0290


Fitting 10 folds for each of 243 candidates, totalling 2430 fits


In [22]:
tuned_grid

ExtraTreesRegressor(max_depth=20, max_features=0.5, n_estimators=600, n_jobs=-1,
                    random_state=3131)

In [25]:
best_params = tuned_grid.get_params()
# Narrow ranges for grid search based on best params again
grid_search_2 = {
    'n_estimators': [max(50, best_params['n_estimators']-100), best_params['n_estimators'], best_params['n_estimators']+100],
    'max_depth': [max(3, best_params['max_depth']-3), best_params['max_depth'], best_params['max_depth']+3],
    'min_samples_split': [max(2, best_params['min_samples_split']-3), best_params['min_samples_split'], best_params['min_samples_split']+3],
    'min_samples_leaf': [max(1, best_params['min_samples_leaf']-2), best_params['min_samples_leaf'], best_params['min_samples_leaf']+2],
    'max_features': [max(0.3, best_params['max_features']-0.1), best_params['max_features'], min(1.0, best_params['max_features']+0.1)],
    'bootstrap': [best_params['bootstrap']]
}

In [26]:
tuned_grid_2 = tune_model(
    et_model,
    optimize='R2',
    search_library='scikit-learn',
    search_algorithm='grid',
    custom_grid=grid_search_2
)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0217,0.0017,0.0413,0.3814,0.0275,0.0437
1,0.0151,0.0004,0.0202,0.7976,0.0128,0.0264
2,0.0168,0.0008,0.0278,0.6191,0.0180,0.0307
3,0.0155,0.0005,0.0216,0.7353,0.0139,0.0275
4,0.0187,0.0007,0.0256,0.6722,0.0167,0.0345
5,0.0117,0.0003,0.0168,0.8059,0.0107,0.0208
6,0.0146,0.0004,0.0208,0.5507,0.0133,0.0256
7,0.0179,0.0012,0.0348,0.4207,0.0231,0.0351
8,0.0171,0.0007,0.0257,0.6070,0.0165,0.0309


Fitting 10 folds for each of 243 candidates, totalling 2430 fits
